# CLEANING ELECTION DATA #
This file works on cleaning the Election dataset prior to both creating variables for the analysis
and adding demographic / text based files

### A) IMPORTS AND SETTINGS ###

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns',200)
pd.set_option('display.width',120)

### B) LOAD RAW FILE

In [59]:
raw_path = 'C:/Users/mutch/Democratic_Voter_Shift/data/processed/election_with_registration.csv' #call file raw_path
df_raw = pd.read_csv(raw_path,dtype=str)  #read all as strings to control parsing
print("Rows,cols:",df_raw.shape)
df_raw.head(5)


Rows,cols: (134, 12)


,county,year,state_po,county_fips,totalvotes,democrat,other,republican,democrat_reg,republican_reg,other_parties_reg,all_parties_reg
0,Adams,2020,PA,42001.0,56540,18207.0,810.0,37523.0,20063.0,40781,11544.0,72388
1,Adams,2024,PA,42001.0,60650,19842.0,560.0,40248.0,19474.0,43914,12620.0,76008
2,Allegheny,2020,PA,42003.0,719733,429065.0,8344.0,282324.0,539595.0,267832,134829.0,942256
3,Allegheny,2024,PA,42003.0,720504,429916.0,6993.0,283595.0,526268.0,274157,152118.0,952543
4,Armstrong,2020,PA,42005.0,36370,8457.0,424.0,27489.0,13659.0,25979,5188.0,44826


### C) NORMALIZE COLUMN NAMES

In [60]:
def clean_col(c):
    c = c.strip().lower().replace(" "," ")
    c=re.sub(r"[^0-9a-z]","",c)
    return c  #this makes all of my variable names very computer readable
df_raw.columns = [clean_col(c) for c in df_raw.columns]
print("Columns:",df_raw.columns.tolist())

Columns: ['county', 'year', 'statepo', 'countyfips', 'totalvotes', 'democrat', 'other', 'republican', 'democratreg', 'republicanreg', 'otherpartiesreg', 'allpartiesreg']


In [61]:
df_raw.head(10)

,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,otherpartiesreg,allpartiesreg
0,Adams,2020,PA,42001.0,56540,18207.0,810.0,37523.0,20063.0,40781,11544.0,72388
1,Adams,2024,PA,42001.0,60650,19842.0,560.0,40248.0,19474.0,43914,12620.0,76008
2,Allegheny,2020,PA,42003.0,719733,429065.0,8344.0,282324.0,539595.0,267832,134829.0,942256
3,Allegheny,2024,PA,42003.0,720504,429916.0,6993.0,283595.0,526268.0,274157,152118.0,952543
4,Armstrong,2020,PA,42005.0,36370,8457.0,424.0,27489.0,13659.0,25979,5188.0,44826
5,Armstrong,2024,PA,42005.0,37119,8553.0,270.0,28296.0,11418.0,27984,5269.0,44671
6,Beaver,2020,PA,42007.0,94122,38122.0,1241.0,54759.0,54281.0,47214,15456.0,116951
7,Beaver,2024,PA,42007.0,94843,37196.0,810.0,56837.0,48284.0,52318,16846.0,117448
8,Bedford,2020,PA,42009.0,27574,4367.0,182.0,23025.0,7264.0,23729,3246.0,34239
9,Bedford,2024,PA,42009.0,28160,4336.0,166.0,23658.0,5671.0,24907,3223.0,33801


### D) CHECK EXPECTED COLUMNS

In [78]:
expected = ["county","year","statepo","countyfips","totalvotes","democrat",
           "other","republican","democratreg","republicanreg","otherpartiesreg",
           "allpartiesreg"]
missing = [c for c in expected if c not in df_raw.columns]
print("Missing expected columns:",missing)

Missing expected columns: []


The above results show that all of the original variable names have been changed.

### E) TRIM WHITESPACE IN STRING COLUMNS

In [63]:
for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df.raw[c].str.strip()

### F) FIX NUMERIC COLUMNS

In [64]:
num_cols = ["countyfips","totalvotes","democrat","other","republican","democratreg",
            "republicanreg","otherpartiesreg","allpartiesreg"]
def clean_numeric_series(s):
    s2 =s.fillna("").astype(str).str.replace(r"[,\s]","",regex=True)
    s2 = s2.str.replace(r"[^\d.-]","",regex=True)  #keep digits, dot, minus
    return pd.to_numeric(s2.replace("",np.nan),errors="coerce")

for c in num_cols:
    if c in df_raw.columns:
        df_raw[c + "_raw"] = df_raw[c] #preserve raw
        df_raw[c] = clean_numeric_series(df_raw[c])
        print(c,"-> nulls:",df_raw[c].isna().sum())
    

countyfips -> nulls: 0
totalvotes -> nulls: 0
democrat -> nulls: 0
other -> nulls: 0
republican -> nulls: 0
democratreg -> nulls: 0
republicanreg -> nulls: 0
otherpartiesreg -> nulls: 0
allpartiesreg -> nulls: 0


In [65]:
df_raw.iloc[60:70]

,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,otherpartiesreg,allpartiesreg,countyfips_raw,totalvotes_raw,democrat_raw,other_raw,republican_raw,democratreg_raw,republicanreg_raw,otherpartiesreg_raw,allpartiesreg_raw
60,Huntingdon,2020,PA,42061.0,22792,5445.0,286.0,17061.0,7470.0,18021,2935.0,28426,42061.0,22792,5445.0,286.0,17061.0,7470.0,18021,2935.0,28426
61,Huntingdon,2024,PA,42061.0,23156,5368.0,161.0,17627.0,6492.0,19202,3034.0,28718,42061.0,23156,5368.0,161.0,17627.0,6492,19202,3034,28718
62,Indiana,2020,PA,42063.0,41198,12634.0,475.0,28089.0,17987.0,27196,6614.0,51797,42063.0,41198,12634.0,475.0,28089.0,17987.0,27196,6614.0,51797
63,Indiana,2024,PA,42063.0,42229,12697.0,317.0,29215.0,15409.0,29541,267.0,51744,42063.0,42229,12697.0,317.0,29215.0,15409.0,"29,541",267.0,"51,744"
64,Jefferson,2020,PA,42065.0,22824,4527.0,337.0,17960.0,8374.0,20008,3658.0,32040,42065.0,22824,4527.0,337.0,17960.0,8374.0,20008,3658.0,32040
65,Jefferson,2024,PA,42065.0,23129,4707.0,187.0,18235.0,5808.0,19750,153.0,28799,42065.0,23129,4707.0,187.0,18235.0,5808.0,"19,750",153.0,"28,799"
66,Juniata,2020,PA,42067.0,12043,2253.0,141.0,9649.0,3364.0,9635,1491.0,14490,42067.0,12043,2253.0,141.0,9649.0,3364.0,9635,1491.0,14490
67,Juniata,2024,PA,42067.0,12104,2290.0,93.0,9721.0,2741.0,10127,51.0,14353,42067.0,12104,2290.0,93.0,9721.0,2741.0,"10,127",51.0,"14,353"
68,Lackawanna,2020,PA,42069.0,115410,61991.0,1085.0,52334.0,85956.0,47059,16374.0,149389,42069.0,115410,61991.0,1085.0,52334.0,85956,47059,16374,149389
69,Lackawanna,2024,PA,42069.0,116590,59510.0,819.0,56261.0,79207.0,52305,660.0,150949,42069.0,116590,59510.0,819.0,56261.0,79207.0,"52,305",660.0,"150,949"


In [66]:
df_raw = df_raw.drop(columns=df_raw.filter(regex=r'_raw$').columns)

In [67]:
df_raw.iloc[60:70]

,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,otherpartiesreg,allpartiesreg
60,Huntingdon,2020,PA,42061.0,22792,5445.0,286.0,17061.0,7470.0,18021,2935.0,28426
61,Huntingdon,2024,PA,42061.0,23156,5368.0,161.0,17627.0,6492.0,19202,3034.0,28718
62,Indiana,2020,PA,42063.0,41198,12634.0,475.0,28089.0,17987.0,27196,6614.0,51797
63,Indiana,2024,PA,42063.0,42229,12697.0,317.0,29215.0,15409.0,29541,267.0,51744
64,Jefferson,2020,PA,42065.0,22824,4527.0,337.0,17960.0,8374.0,20008,3658.0,32040
65,Jefferson,2024,PA,42065.0,23129,4707.0,187.0,18235.0,5808.0,19750,153.0,28799
66,Juniata,2020,PA,42067.0,12043,2253.0,141.0,9649.0,3364.0,9635,1491.0,14490
67,Juniata,2024,PA,42067.0,12104,2290.0,93.0,9721.0,2741.0,10127,51.0,14353
68,Lackawanna,2020,PA,42069.0,115410,61991.0,1085.0,52334.0,85956.0,47059,16374.0,149389
69,Lackawanna,2024,PA,42069.0,116590,59510.0,819.0,56261.0,79207.0,52305,660.0,150949


### G) CONVERT YEAR & COUNTYFIPS TYPES.  STANDARDIZE COUNTY NAMES

In [74]:
if "year" in df_raw.columns:
    df_raw["year"] = pd.to_numeric(df_raw["year"], errors="coerce").astype("Int64")

if "countyfips" in df_raw.columns:
    df_raw["countyfips"] = df_raw["countyfips"].astype(str).str.strip()
    df_raw["countyfips"] = df_raw["countyfips"].str.replace(r"\.0$", "", regex=True)
    df_raw["countyfips"] = df_raw["countyfips"].str.zfill(5)

df_raw["county"] = df_raw["county"].str.title().str.replace(r"\s+", " ", regex=True).str.strip()



In [75]:
df_raw.dtypes

county                 str
year                 Int64
statepo                str
countyfips             str
totalvotes           int64
democrat           float64
other              float64
republican         float64
democratreg        float64
republicanreg        int64
otherpartiesreg    float64
allpartiesreg        int64
dtype: object

### H) CHECK MISSINGNESS OR DUPLICATION AND REPORT

In [79]:
print("Missingness (per column):")
print(df_raw[expected].isna().sum())
print("\nFull-row duplicates:",df_raw.duplicated().sum())
print("County-Year Duplicates:",df_raw.duplicated(subset=["county","year"]).sum())

Missingness (per column):
county             0
year               0
statepo            0
countyfips         0
totalvotes         0
democrat           0
other              0
republican         0
democratreg        0
republicanreg      0
otherpartiesreg    0
allpartiesreg      0
dtype: int64

Full-row duplicates: 0
County-Year Duplicates: 0


The results above show no missing or duplicates in any column

### I) CHECK INTERNAL CONSISTENCY

In [86]:
# Fixed: Changed 'df' to 'df_raw' to match the variable used throughout the code
check_cols_votes = [c for c in ["democrat","republican","other"] if c in df_raw.columns]
df_raw["votes_sum"] = df_raw[check_cols_votes].sum(axis=1, skipna=True)
df_raw["votes_total_diff"] = df_raw["totalvotes"] - df_raw["votes_sum"]
print("Votes diff summary (totalvotes - sum candidates):")
print(df_raw["votes_total_diff"].describe())

check_cols_regs = [c for c in ["democrat_reg","republican_reg","other_parties_reg"] if c in df_raw.columns]
if check_cols_regs:
    df_raw["regs_sum"] = df_raw[check_cols_regs].sum(axis=1, skipna=True)
    df_raw["regs_total_diff"] = df_raw["all_parties_reg"] - df_raw["regs_sum"]
    print("Regs diff summary (all_parties_reg - sum regs):")
    print(df_raw["regs_total_diff"].describe())

# Flag large mismatches
df_raw["flag_votes_mismatch"] = df_raw["votes_total_diff"].abs() > 1 # threshold 1 vote; adjust as needed
# Fixed: Added proper handling for when regs_total_diff column doesn't exist
df_raw["flag_regs_mismatch"] = df_raw.get("regs_total_diff", pd.Series(0, index=df_raw.index)).abs() > 1

print("Rows with votes mismatch:", df_raw["flag_votes_mismatch"].sum())
333print("Rows with regs mismatch:", df_raw["flag_regs_mismatch"].sum())

Votes diff summary (totalvotes - sum candidates):
count    134.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: votes_total_diff, dtype: float64
Rows with votes mismatch: 0
Rows with regs mismatch: 0


**saving the df_raw as 'election_base' in the /data/final/ file directory** 

In [87]:
df_raw.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/final/election_base.csv")

*Saving again as a backup pure file*

In [88]:
df_raw.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/final/election_base_DoNotTouch.csv")

### J) PART 2: CREATING ANALYSIS VARIABLES

**From the table on pg.6 of prospectus,the main analytical variables are:**
- dem_vote_share.
- gop_vote_share.
- turnout_rate.
- dem_vote_share_2020.
- dem_vote_share_2024.
- dem_vote_share_change.
- turnout_change.
- shift_away_from_dem.
- shift_category.


**2A) IMPORT**

In [ ]:
import pandas as pd
import numpy as np

In [90]:
df_analytical = pd.read_csv("C:/Users/mutch/Democratic_Voter_Shift/data/final/election_base.csv")
df_analytical.head(7)

,Unnamed: 0,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,otherpartiesreg,allpartiesreg,votes_sum,votes_total_diff,flag_votes_mismatch,flag_regs_mismatch
0,0,Adams,2020,PA,42001,56540,18207.0,810.0,37523.0,20063.0,40781,11544.0,72388,56540.0,0.0,False,False
1,1,Adams,2024,PA,42001,60650,19842.0,560.0,40248.0,19474.0,43914,12620.0,76008,60650.0,0.0,False,False
2,2,Allegheny,2020,PA,42003,719733,429065.0,8344.0,282324.0,539595.0,267832,134829.0,942256,719733.0,0.0,False,False
3,3,Allegheny,2024,PA,42003,720504,429916.0,6993.0,283595.0,526268.0,274157,152118.0,952543,720504.0,0.0,False,False
4,4,Armstrong,2020,PA,42005,36370,8457.0,424.0,27489.0,13659.0,25979,5188.0,44826,36370.0,0.0,False,False
5,5,Armstrong,2024,PA,42005,37119,8553.0,270.0,28296.0,11418.0,27984,5269.0,44671,37119.0,0.0,False,False
6,6,Beaver,2020,PA,42007,94122,38122.0,1241.0,54759.0,54281.0,47214,15456.0,116951,94122.0,0.0,False,False


In [91]:
df_analytical.columns

Index(['Unnamed: 0', 'county', 'year', 'statepo', 'countyfips', 'totalvotes', 'democrat', 'other', 'republican',
       'democratreg', 'republicanreg', 'otherpartiesreg', 'allpartiesreg', 'votes_sum', 'votes_total_diff',
       'flag_votes_mismatch', 'flag_regs_mismatch'],
      dtype='str')

In [96]:
df_analytical = df_analytical.drop(columns=["flag_regs_mismatch"])
df_analytical.head(10)

,Unnamed: 0,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,allpartiesreg
0,0,Adams,2020,PA,42001,56540,18207.0,810.0,37523.0,20063.0,40781,72388
1,1,Adams,2024,PA,42001,60650,19842.0,560.0,40248.0,19474.0,43914,76008
2,2,Allegheny,2020,PA,42003,719733,429065.0,8344.0,282324.0,539595.0,267832,942256
3,3,Allegheny,2024,PA,42003,720504,429916.0,6993.0,283595.0,526268.0,274157,952543
4,4,Armstrong,2020,PA,42005,36370,8457.0,424.0,27489.0,13659.0,25979,44826
5,5,Armstrong,2024,PA,42005,37119,8553.0,270.0,28296.0,11418.0,27984,44671
6,6,Beaver,2020,PA,42007,94122,38122.0,1241.0,54759.0,54281.0,47214,116951
7,7,Beaver,2024,PA,42007,94843,37196.0,810.0,56837.0,48284.0,52318,117448
8,8,Bedford,2020,PA,42009,27574,4367.0,182.0,23025.0,7264.0,23729,34239
9,9,Bedford,2024,PA,42009,28160,4336.0,166.0,23658.0,5671.0,24907,33801


In [97]:
df_analytical.dtypes

Unnamed: 0         int64
county               str
year               int64
statepo              str
countyfips         int64
totalvotes         int64
democrat         float64
other            float64
republican       float64
democratreg      float64
republicanreg      int64
allpartiesreg      int64
dtype: object

**2B) CREATE VOTESHARE AND TURNOUT VARIABLES** 

In [99]:
df_analytical["demvoteshare"] = df_analytical["democrat"] / df_analytical["totalvotes"]
df_analytical["gopvoteshare"] = df_analytical["republican"] / df_analytical["totalvotes"]
df_analytical["othervoteshare"] = df_analytical["other"] / df_analytical["totalvotes"]

df_analytical["turnoutrate"] = df_analytical["totalvotes"] / df_analytical["allpartiesreg"]

In [100]:
df_analytical.head(6)

,Unnamed: 0,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,allpartiesreg,demvoteshare,gopvoteshare,othervoteshare,turnoutrate
0,0,Adams,2020,PA,42001,56540,18207.0,810.0,37523.0,20063.0,40781,72388,0.322020,0.663654,0.014326,0.781069
1,1,Adams,2024,PA,42001,60650,19842.0,560.0,40248.0,19474.0,43914,76008,0.327156,0.663611,0.009233,0.797942
2,2,Allegheny,2020,PA,42003,719733,429065.0,8344.0,282324.0,539595.0,267832,942256,0.596145,0.392262,0.011593,0.763840
3,3,Allegheny,2024,PA,42003,720504,429916.0,6993.0,283595.0,526268.0,274157,952543,0.596688,0.393606,0.009706,0.756400
4,4,Armstrong,2020,PA,42005,36370,8457.0,424.0,27489.0,13659.0,25979,44826,0.232527,0.755815,0.011658,0.811359
5,5,Armstrong,2024,PA,42005,37119,8553.0,270.0,28296.0,11418.0,27984,44671,0.230421,0.762305,0.007274,0.830942


In [102]:
df_analytical.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/final/pickuptuesday.csv")

In [3]:
pickup = ("C:/Users/mutch/Democratic_Voter_Shift/data/final/pickuptuesday.csv")
df = pd.read_csv(pickup)
df.head()

,Unnamed: 0.1,Unnamed: 0,county,year,statepo,countyfips,totalvotes,democrat,other,republican,democratreg,republicanreg,allpartiesreg,demvoteshare,gopvoteshare,othervoteshare,turnoutrate
0,0,0,Adams,2020,PA,42001,56540,18207.0,810.0,37523.0,20063.0,40781,72388,0.322020,0.663654,0.014326,0.781069
1,1,1,Adams,2024,PA,42001,60650,19842.0,560.0,40248.0,19474.0,43914,76008,0.327156,0.663611,0.009233,0.797942
2,2,2,Allegheny,2020,PA,42003,719733,429065.0,8344.0,282324.0,539595.0,267832,942256,0.596145,0.392262,0.011593,0.763840
3,3,3,Allegheny,2024,PA,42003,720504,429916.0,6993.0,283595.0,526268.0,274157,952543,0.596688,0.393606,0.009706,0.756400
4,4,4,Armstrong,2020,PA,42005,36370,8457.0,424.0,27489.0,13659.0,25979,44826,0.232527,0.755815,0.011658,0.811359


**2C) reshape to wide format**

Each county will now have 1 row, with columns representing 2020 and 2024 variables.

In [4]:
wide = df.pivot_table(
    index=["county", "countyfips", "statepo"],
    columns="year",
    values=["demvoteshare", "gopvoteshare", "turnoutrate", "totalvotes"],
    aggfunc="first"
)

wide.columns = [f"{var}_{yr}" for var, yr in wide.columns]
wide = wide.reset_index()


In [6]:
wide.head()

,county,countyfips,statepo,demvoteshare_2020,demvoteshare_2024,gopvoteshare_2020,gopvoteshare_2024,totalvotes_2020,totalvotes_2024,turnoutrate_2020,turnoutrate_2024
0,Adams,42001,PA,0.322020,0.327156,0.663654,0.663611,56540,60650,0.781069,0.797942
1,Allegheny,42003,PA,0.596145,0.596688,0.392262,0.393606,719733,720504,0.763840,0.756400
2,Armstrong,42005,PA,0.232527,0.230421,0.755815,0.762305,36370,37119,0.811359,0.830942
3,Beaver,42007,PA,0.405028,0.392185,0.581787,0.599275,94122,94843,0.804799,0.807532
4,Bedford,42009,PA,0.158374,0.153977,0.835026,0.840128,27574,28160,0.805339,0.833111


**COMPUTING CHANGES IN VOTE SHARES & TURNOUT RATES**

In [7]:
if "demvoteshare_2020" in wide.columns and "demvoteshare_2024" in wide.columns:
    wide["demvoteshare_change"] = wide["demvoteshare_2024"] - wide["demvoteshare_2020"]
else:
    wide["demvoteshare_change"] = np.nan

if "gopvoteshare_2020" in wide.columns and "gopvoteshare_2024" in wide.columns:
    wide["gopvoteshare_change"] = wide["gopvoteshare_2024"] - wide["gopvoteshare_2020"]
else:
    wide["gopvoteshare_change"] = np.nan

if "turnoutrate_2020" in wide.columns and "turnoutrate_2024" in wide.columns:
    wide["turnout_change"] = wide["turnoutrate_2024"] - wide["turnoutrate_2020"]
else:
    wide["turnout_change"] = np.nan

wide.head()


,county,countyfips,statepo,demvoteshare_2020,demvoteshare_2024,gopvoteshare_2020,gopvoteshare_2024,totalvotes_2020,totalvotes_2024,turnoutrate_2020,turnoutrate_2024,demvoteshare_change,gopvoteshare_change,turnout_change
0,Adams,42001,PA,0.322020,0.327156,0.663654,0.663611,56540,60650,0.781069,0.797942,0.005136,-0.000043,0.016874
1,Allegheny,42003,PA,0.596145,0.596688,0.392262,0.393606,719733,720504,0.763840,0.756400,0.000543,0.001344,-0.007440
2,Armstrong,42005,PA,0.232527,0.230421,0.755815,0.762305,36370,37119,0.811359,0.830942,-0.002106,0.006490,0.019582
3,Beaver,42007,PA,0.405028,0.392185,0.581787,0.599275,94122,94843,0.804799,0.807532,-0.012843,0.017487,0.002733
4,Bedford,42009,PA,0.158374,0.153977,0.835026,0.840128,27574,28160,0.805339,0.833111,-0.004397,0.005102,0.027773


In [8]:
wide.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/election_base_f.csv")